# Suite de Evaluación Cuantitativa
Ejecución de un *Ground Truth Dataset* automatizado para calcular el **Hit Rate** y la eficacia del umbral de seguridad de nuestro sistema multiagente y RAG Híbrido.

In [1]:
import sys
import os
from pathlib import Path

# Alineación de rutas para ejecución independiente
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [2]:
import time
from typing import List
from src.rag_pipeline import EntityResolutionPipeline

class RAGEvaluator:
    def __init__(self):
        print("🔌 Inicializando motor de evaluación RAG...")
        self.pipeline = EntityResolutionPipeline()
        
        self.dataset = [
            {"query": "ANTARIUS S.A.", "exp_match": True, "exp_review": False, "cat": "Ruido Leve"},
            {"query": "Al Sariya Commercial Investments L.L.C.", "exp_match": True, "exp_review": False, "cat": "Ruido Leve"},
            {"query": "Alecta pensionsforsakring, omsesidigt", "exp_match": True, "exp_review": False, "cat": "Falta Diéresis"},
            {"query": "BNP Paribas", "exp_match": True, "exp_review": False, "cat": "Match Exacto"},
            {"query": "DHFI 13 FZ LLC", "exp_match": True, "exp_review": False, "cat": "Ruido Leve"},
            {"query": "Appia IV Global Infrastructure Portfolio SCSp", "exp_match": False, "exp_review": True, "cat": "Serie Distinta (IV vs III)"},
            {"query": "Border to Coast Bedfordshire Fund II LP", "exp_match": False, "exp_review": True, "cat": "Serie Distinta (Fund II)"},
            {"query": "Caisse Regionale Crédit Agricole Mutuel de Paris", "exp_match": False, "exp_review": True, "cat": "Geografía Distinta (Paris)"},
            {"query": "Border to Coast London LP", "exp_match": False, "exp_review": True, "cat": "Geografía Distinta (London)"},
            {"query": "DHFI 14 FZ-LLC", "exp_match": False, "exp_review": True, "cat": "Serie Distinta (14 vs 13)"},
            {"query": "Tech Ventures Capital Innovacion SA", "exp_match": False, "exp_review": True, "cat": "Inexistente"},
            {"query": "Global Horizon Partners L.P.", "exp_match": False, "exp_review": True, "cat": "Inexistente"},
            {"query": "Banco Santander Investment Hub", "exp_match": False, "exp_review": True, "cat": "Inexistente"},
            {"query": "NextGen Renewable Energy Fund", "exp_match": False, "exp_review": True, "cat": "Inexistente"},
            {"query": "Omega Quantitative Solutions LLC", "exp_match": False, "exp_review": True, "cat": "Inexistente"}
        ]

    def run_evaluation(self):
        print(f"\n🚀 Iniciando Evaluación sobre {len(self.dataset)} casos de prueba...")
        print("="*80)
        
        match_correctos = 0
        seguridad_correcta = 0
        tiempos = []
        
        for i, case in enumerate(self.dataset, 1):
            query = case["query"]
            exp_match = case["exp_match"]
            exp_review = case["exp_review"]
            categoria = case["cat"]
            
            print(f"[{i}/{len(self.dataset)}] Testeando ({categoria}): '{query}'")
            
            start_time = time.time()
            try:
                result = self.pipeline.chain.invoke(query)
                tiempos.append(time.time() - start_time)
                
                pred_match = result.get("is_match", False)
                pred_review = result.get("trigger_human_review", True)
                
                is_match_correct = (pred_match == exp_match)
                is_safety_correct = (pred_review == exp_review)
                
                if is_match_correct: match_correctos += 1
                if is_safety_correct: seguridad_correcta += 1
                
                status = "✅ PASS" if (is_match_correct and is_safety_correct) else "❌ FAIL"
                print(f"   ↳ {status} | Match Esperado: {exp_match}(Pred: {pred_match}) | Revisión Esperada: {exp_review}(Pred: {pred_review})")
                
            except Exception as e:
                print(f"   ↳ ⚠️ ERROR: {str(e)}")
        
        self._print_dashboard(match_correctos, seguridad_correcta, tiempos)

    def _print_dashboard(self, m_corr, s_corr, tiempos):
        total = len(self.dataset)
        print("\n" + "="*80)
        print(" 📊 DASHBOARD DE MÉTRICAS - ENTITY RESOLUTION PIPELINE")
        print("="*80)
        print(f" 📈 Hit Rate (Precisión de Match):        {(m_corr/total)*100:.1f}% ({m_corr}/{total})")
        print(f" 🛡️ Safety Rate (Precisión de Seguridad):  {(s_corr/total)*100:.1f}% ({s_corr}/{total})")
        print(f" ⏱️ Latencia Media por Query:              {sum(tiempos)/len(tiempos):.2f} segundos")
        print("="*80)

# Ejecutar la clase
evaluator = RAGEvaluator()
evaluator.run_evaluation()

/home/ema/Documentos/master/genAI_project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔌 Inicializando motor de evaluación RAG...
🔌 Inicializando LLM: Groq (llama-3.3-70b-versatile)
⚠️ AVISO: No se detectó OPENAI_API_KEY. Fallback a Embeddings Locales de HuggingFace.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4208.77it/s]



🚀 Iniciando Evaluación sobre 15 casos de prueba...
[1/15] Testeando (Ruido Leve): 'ANTARIUS S.A.'


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


...

   ↳ ❌ FAIL | Match Esperado: False(Pred: True) | Revisión Esperada: True(Pred: False)
[14/15] Testeando (Inexistente): 'NextGen Renewable Energy Fund'


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 5.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


   ↳ ✅ PASS | Match Esperado: False(Pred: False) | Revisión Esperada: True(Pred: True)
[15/15] Testeando (Inexistente): 'Omega Quantitative Solutions LLC'


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 4.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 1.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


   ↳ ✅ PASS | Match Esperado: False(Pred: False) | Revisión Esperada: True(Pred: True)

 📊 DASHBOARD DE MÉTRICAS - ENTITY RESOLUTION PIPELINE
 📈 Hit Rate (Precisión de Match):        93.3% (14/15)
 🛡️ Safety Rate (Precisión de Seguridad):  93.3% (14/15)
 ⏱️ Latencia Media por Query:              1.81 segundos
